# 02b — Debug and Fix Spectral Retrieval
### Enveda CASMI 2026 — verifying and correcting Notebook 02's spectral baselines

**Why this notebook exists.** Notebook 02's B2-B6 spectral/hybrid baselines scored materially
*worse* than the mass-only B1 baseline (MRR@25 ~ 0.295), and B2 through B6 looked "wired
identically" -- ion-mode/adduct/CE conditioning barely moved the numbers. That is not the
expected signature of "spectral similarity doesn't help"; it is the signature of a **ranking
bug**. This notebook diagnoses it precisely, fixes it, and re-verifies every component before
anything downstream (Notebook 03) is allowed to build on top of it.

**Diagnosis (confirmed by re-reading notebook 02's actual code, `cosine_retrieve_hier`):**

```python
ranked_with_ref = [k for k, _ in sorted(scored.items(), key=lambda kv: -kv[1])]
remaining = [k for k in candidate_keys if k not in scored]
return (ranked_with_ref + remaining)[:25]
```

Every candidate that has *any* reference spectrum in the fold-complement pool is placed ahead of
every candidate that doesn't -- **regardless of how weak the cosine score is, or how good the
mass fit is.** Notebook 02 Section 6 already established that, under connectivity-grouped CV, a
query's own true connectivity has **zero** reference spectra in the fold-complement pool by
construction. So the true candidate was almost always dumped into the `remaining` (mass-order)
bucket, and placed **after every wrong candidate that happened to have a reference spectrum** --
however poor its mass fit. That single hard partition explains both symptoms: (a) B2-B6 scoring
below mass-only B1 (mass fit alone would have ranked the true candidate much higher), and (b)
B2-B6 looking nearly identical to each other (ion/adduct/CE conditioning only reshuffles the
small reference-bearing subset; it never touches the dominant partition).

**The fix**, implemented once in `src/retrieval_baseline.py` (not re-implemented here or in
Notebook 03): `hybrid_candidate_score` -- an *additive* blend where mass fit always contributes
and spectral evidence adds to it, rather than gating candidates into a hard partition.

This notebook verifies, in order: cosine direction, aggregation correctness, multimer/dimer mass
arithmetic, `target_in_reference_library`/novel-connectivity label consistency, the corrected
hybrid scorer's ranking behaviour, and finally re-measures MRR@25 across all 5 folds against the
mass-only reference.


## 1. Imports and loading Notebook 02's verified artifacts

We reuse Notebook 02's structure library, connectivity/fold assignment, and cached spectral
vectors rather than recomputing them (they are not what's broken). Only the *scoring/ranking*
logic is being fixed here.


In [ ]:
import sys, time
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import sparse

SEED = 42
np.random.seed(SEED)
pd.set_option("display.max_columns", 200)
plt.rcParams["figure.figsize"] = (9, 5)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
BASELINE_DIR = PROJECT_ROOT / "outputs" / "baseline"
OUT_DIR = PROJECT_ROOT / "outputs" / "baseline_debug"
OUT_DIR.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src import chemistry, spectra as spec_utils, validation, metrics as metrics_mod, retrieval_baseline as rb

print("Imported src.retrieval_baseline from:", rb.MODULE_PATH)
print("Module content hash:", rb.module_version_hash())

REQUIRED_ARTIFACTS = [
    BASELINE_DIR / "structure_library.parquet",
    BASELINE_DIR / "connectivity_folds.parquet",
    BASELINE_DIR / "train_spectral_vectors.npz",
    BASELINE_DIR / "test_spectral_vectors.npz",
]
missing = [p for p in REQUIRED_ARTIFACTS if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Notebook 02 has not been run (or its outputs were cleared). Missing:\n"
        + "\n".join(f"  - {p}" for p in missing)
        + "\n\nRun notebooks/02_baseline_mass_spectral_retrieval.ipynb top-to-bottom first; "
          "this debug notebook verifies and fixes its scoring logic, it does not regenerate "
          "the structure library / folds / spectral vectors from scratch."
    )
print("\nAll required Notebook 02 artifacts found.")


In [ ]:
train_df = pd.read_parquet(DATA_DIR / "train.parquet")
test_df = pd.read_parquet(DATA_DIR / "test.parquet")
if "spectrum_id" not in train_df.columns:
    train_df["spectrum_id"] = "train_" + train_df.index.astype(str)
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

structure_library = pd.read_parquet(BASELINE_DIR / "structure_library.parquet")
connectivity_folds = pd.read_parquet(BASELINE_DIR / "connectivity_folds.parquet")
train_vectors = sparse.load_npz(BASELINE_DIR / "train_spectral_vectors.npz")
test_vectors = sparse.load_npz(BASELINE_DIR / "test_spectral_vectors.npz")

assert train_vectors.shape[0] == len(train_df), "Cached vectors no longer align with train_df row order."

# DATA_MAP: same resolution as Notebook 02 (kept minimal here -- 02 already validated this).
smiles_col, adduct_col, ion_col = "normalized_smiles", "adduct", "ionization_mode"
mz_col, peaks_col, peaks_intensity_col = "precursor_mz", "ms2_mzs", "ms2_normalized_intensities"
ce_col, spec_col, test_mol_col = "collision_energy_ev", "spectrum_id", "molecule_id"

# Re-attach connectivity_key + fold from the SAVED fold assignment (source of truth), not recomputed.
fold_map = connectivity_folds.drop_duplicates(spec_col).set_index(spec_col)
train_df["connectivity_key"] = train_df[spec_col].map(fold_map["connectivity_key"])
train_df["fold"] = train_df[spec_col].map(fold_map["fold"])
print(f"Rows with a fold assignment: {train_df['fold'].notna().sum():,} / {len(train_df):,}")

# Adduct table + neutral mass (identical logic to Notebook 02, now sourced from src/retrieval_baseline).
adduct_table = rb.build_adduct_table(pd.concat([train_df[adduct_col], test_df[adduct_col]]))

def attach_neutral_mass(df):
    info = df[adduct_col].map(adduct_table.to_dict("index"))
    supported = info.map(lambda d: isinstance(d, dict) and d["supported"])
    z = info.map(lambda d: d["z"] if isinstance(d, dict) else np.nan).astype(float)
    n = info.map(lambda d: d["n"] if isinstance(d, dict) else np.nan).astype(float)
    delta = info.map(lambda d: d["delta"] if isinstance(d, dict) else np.nan).astype(float)
    electron_term = info.map(lambda d: d["electron_term"] if isinstance(d, dict) else np.nan).astype(float)
    df = df.copy()
    df["adduct_supported"] = supported.astype(bool)
    neutral_mass = (df[mz_col].astype(float) * z - delta - electron_term) / n
    df["neutral_mass_estimate"] = np.where(supported, neutral_mass, np.nan)
    return df

train_df = attach_neutral_mass(train_df)
test_df = attach_neutral_mass(test_df)
print("neutral_mass_estimate coverage:", train_df["neutral_mass_estimate"].notna().mean())

DEV_FOLD = 0
foldable = train_df["fold"].notna()
train_mask = foldable & (train_df["fold"] != DEV_FOLD)
valid_mask = train_df["fold"] == DEV_FOLD
train_mask_idx = np.flatnonzero(train_mask.to_numpy())
ref_connectivity = train_df["connectivity_key"].to_numpy()[train_mask_idx]
conn_to_positions = defaultdict(list)
for pos, key in enumerate(ref_connectivity):
    if isinstance(key, str):
        conn_to_positions[key].append(pos)
ref_vectors = train_vectors[train_mask_idx]
mass_index = rb.MassCandidateIndex(structure_library)
print(f"\nDEV_FOLD={DEV_FOLD}: reference pool {ref_vectors.shape[0]:,} spectra, "
      f"{len(conn_to_positions):,} distinct connectivities.")


## 2. Spectral self-similarity / cosine-direction sanity check

A spectrum compared against **itself** must score ~1.0 (vectors are L2-normalized, so the dot
product of a vector with itself is exactly its squared norm = 1.0). A spectrum compared against
an unrelated, randomly chosen spectrum must score much lower. If either fails, the vectors are
built or dotted with the wrong orientation/normalization.


In [ ]:
rng = np.random.default_rng(SEED)
sample_pos = rng.choice(train_vectors.shape[0], size=200, replace=False)

self_sim = np.array([(train_vectors[p] @ train_vectors[p].T).toarray().item() for p in sample_pos])
other_pos = rng.permutation(sample_pos)
random_sim = np.array([(train_vectors[a] @ train_vectors[b].T).toarray().item()
                        for a, b in zip(sample_pos, other_pos) if a != b])

cosine_sanity = pd.DataFrame({
    "check": ["self_similarity", "random_pair_similarity"],
    "mean": [self_sim.mean(), random_sim.mean()],
    "min": [self_sim.min(), random_sim.min()],
    "max": [self_sim.max(), random_sim.max()],
    "n": [len(self_sim), len(random_sim)],
})
display(cosine_sanity)

SELF_SIM_OK = bool(np.allclose(self_sim, 1.0, atol=1e-6))
DIRECTION_OK = bool(self_sim.mean() > random_sim.mean() + 0.3)
print(f"\nself-similarity ~= 1.0 for every sampled spectrum: {SELF_SIM_OK}")
print(f"self-similarity >> random-pair similarity (by a wide margin): {DIRECTION_OK} "
      f"({self_sim.mean():.4f} vs {random_sim.mean():.4f})")
cosine_sanity.to_csv(OUT_DIR / "cosine_sanity.csv", index=False)


## 3. Aggregation audit

Verify `aggregate_reference_scores` behaves correctly against a synthetic ground truth, then
against a real candidate with several reference spectra.


In [ ]:
synthetic = np.array([0.1, 0.9, 0.5, 0.3, 0.95, 0.2, 0.05])
expected = {
    "max": float(synthetic.max()),
    "mean": float(synthetic.mean()),
    "top3": float(np.sort(synthetic)[::-1][:3].mean()),
    "top5": float(np.sort(synthetic)[::-1][:5].mean()),
}
agg_audit_rows = []
for method, exp in expected.items():
    got = rb.aggregate_reference_scores(synthetic, method)
    agg_audit_rows.append({"method": method, "expected": exp, "got": got, "match": np.isclose(exp, got)})
    assert np.isclose(exp, got), f"{method} aggregation mismatch: expected {exp}, got {got}"

# Ordering sanity that must ALWAYS hold for any nonneg score set: max >= top3 >= mean, top5 >= mean.
assert expected["max"] >= expected["top3"] >= expected["mean"]
agg_audit_rows.append({"method": "ordering_max>=top3>=mean", "expected": True,
                        "got": expected["max"] >= expected["top3"] >= expected["mean"], "match": True})

# Real example: pick a connectivity with >=5 reference spectra in the DEV_FOLD pool.
sizes = {k: len(v) for k, v in conn_to_positions.items()}
big_key = max(sizes, key=sizes.get)
pos_list = np.asarray(conn_to_positions[big_key])
query_pos = pos_list[0]  # use one of its own spectra as a stand-in query for this audit only
sims_real = (ref_vectors[pos_list] @ train_vectors[train_mask_idx[query_pos]].T).toarray().ravel()
for method in ("max", "mean", "top3", "top5"):
    agg_audit_rows.append({"method": f"real_{method}", "expected": np.nan,
                            "got": rb.aggregate_reference_scores(sims_real, method), "match": True})

aggregation_audit = pd.DataFrame(agg_audit_rows)
display(aggregation_audit)
AGGREGATION_OK = bool(aggregation_audit["match"].all())
print(f"\nAggregation audit passed: {AGGREGATION_OK}")
aggregation_audit.to_csv(OUT_DIR / "aggregation_audit.csv", index=False)


## 4. Multimer / dimer neutral-mass formula validation

For each of `[2M+H]+`, `[2M+Na]+`, `[2M-H]-` (and a couple of monomer adducts as a control), we
take a **real** example from the data where available, compute the inferred neutral mass from
`precursor_mz` + adduct, and compare against the **known** exact mass of that row's own
structure (from `structure_library`, keyed by the row's own `connectivity_key` -- this is
independent of retrieval, a pure chemistry-arithmetic check). Where a multimer adduct does not
actually occur in this dataset, we fall back to a hand-computed synthetic case so the formula is
still exercised and clearly labeled as such.


In [ ]:
TARGET_ADDUCTS = ["[2M+H]+", "[2M+Na]+", "[2M-H]-", "[M+H]+", "[M-H]-"]
key_to_mass = structure_library.set_index("connectivity_key")["exact_mass"]

multimer_rows = []
for add in TARGET_ADDUCTS:
    subset = train_df[(train_df[adduct_col] == add) & train_df["neutral_mass_estimate"].notna()
                       & train_df["connectivity_key"].notna()]
    if len(subset):
        row = subset.iloc[0]
        true_mass = key_to_mass.get(row["connectivity_key"])
        if true_mass is not None and np.isfinite(true_mass):
            ppm_err = 1e6 * (row["neutral_mass_estimate"] - true_mass) / true_mass
            multimer_rows.append({"adduct": add, "source": "real", "precursor_mz": row[mz_col],
                                   "inferred_neutral_mass": row["neutral_mass_estimate"],
                                   "known_exact_mass": true_mass, "ppm_error": ppm_err})
            continue
    # Synthetic fallback: hand-computed from a real small molecule (glucose, C6H12O6, exact mass 180.0634).
    glucose_mass = 180.06339
    parsed = chemistry.parse_adduct(add)
    if parsed is None:
        multimer_rows.append({"adduct": add, "source": "synthetic_unsupported", "precursor_mz": np.nan,
                               "inferred_neutral_mass": np.nan, "known_exact_mass": glucose_mass, "ppm_error": np.nan})
        continue
    delta = sum(sign * count * chemistry.formula_mass(frag) for sign, count, frag in parsed["terms"])
    z, n = parsed["z"], parsed["n"]
    electron_term = -z * chemistry.ELECTRON_MASS if parsed["polarity"] == "+" else z * chemistry.ELECTRON_MASS
    synthetic_precursor = (glucose_mass * n + delta + electron_term) / z
    inferred = chemistry.neutral_mass_from_precursor(synthetic_precursor, add)
    ppm_err = 1e6 * (inferred - glucose_mass) / glucose_mass
    multimer_rows.append({"adduct": add, "source": "synthetic_glucose", "precursor_mz": synthetic_precursor,
                           "inferred_neutral_mass": inferred, "known_exact_mass": glucose_mass, "ppm_error": ppm_err})

multimer_validation = pd.DataFrame(multimer_rows)
display(multimer_validation)

MULTIMER_OK = bool((multimer_validation["ppm_error"].abs() < 1.0).all())  # sub-1ppm: pure arithmetic, should be near-exact
print(f"\nAll |ppm_error| < 1.0 ppm (pure formula-reconstruction check, not instrument error): {MULTIMER_OK}")
multimer_validation.to_csv(OUT_DIR / "multimer_mass_validation.csv", index=False)


## 5. `target_in_reference_library` / `novel_connectivity_ceiling` consistency

Guardrail: no query may simultaneously be `novel_connectivity_ceiling == True` (no reference
spectra available at all) and `target_in_reference_library == True` (a reference spectrum exists)
-- these are logical complements by definition and must never both hold.


In [ ]:
valid_conn = train_df.loc[valid_mask & train_df["connectivity_key"].notna(), "connectivity_key"]
target_in_reference_library = valid_conn.map(lambda k: k in conn_to_positions)
novel_connectivity_ceiling = ~target_in_reference_library

overlap = (target_in_reference_library & novel_connectivity_ceiling).sum()
known_vs_novel_results = pd.DataFrame({
    "label": ["target_in_reference_library", "novel_connectivity_ceiling"],
    "true_count": [int(target_in_reference_library.sum()), int(novel_connectivity_ceiling.sum())],
    "fraction": [float(target_in_reference_library.mean()), float(novel_connectivity_ceiling.mean())],
})
display(known_vs_novel_results)
print(f"\nSimultaneous True/True (must be 0): {overlap}")
LABEL_CONSISTENCY_OK = bool(overlap == 0) and bool(np.isclose(
    target_in_reference_library.mean() + novel_connectivity_ceiling.mean(), 1.0))
print(f"Labels are exact logical complements covering 100% of queries: {LABEL_CONSISTENCY_OK}")
assert LABEL_CONSISTENCY_OK, "target_in_reference_library and novel_connectivity_ceiling are inconsistent!"

print(f"\ntarget_in_reference_library rate on DEV_FOLD: {target_in_reference_library.mean():.4%} "
      "(expected ~0% under connectivity-grouped CV -- confirms Notebook 02 Section 6's finding).")
known_vs_novel_results.to_csv(OUT_DIR / "label_consistency_check.csv", index=False)
# The fuller known_vs_novel_results.csv (actual MRR/Hit@k split by this label) is written in
# Section 9, once the corrected hybrid scorer's rankings are available.


## 6. Reproducing Notebook 02's bug empirically

Reimplemented here **only to demonstrate the failure mode being fixed** (this is not a second
production implementation -- Section 7 switches to the corrected `src/retrieval_baseline`
scorer and everything downstream uses that).


In [ ]:
def old_buggy_partition_rank(candidate_keys, sims_by_key):
    """Exact reproduction of Notebook 02's `cosine_retrieve_hier` return logic."""
    scored = {k: v for k, v in sims_by_key.items() if k in candidate_keys}
    ranked_with_ref = [k for k, _ in sorted(scored.items(), key=lambda kv: -kv[1])]
    remaining = [k for k in candidate_keys if k not in scored]
    return (ranked_with_ref + remaining)[:25]

def score_candidates_cosine(query_vec, candidate_keys, agg="max"):
    sims_by_key = {}
    for key in candidate_keys:
        pos_list = conn_to_positions.get(key)
        if not pos_list:
            continue
        sims = (ref_vectors[np.asarray(pos_list)] @ query_vec.T).toarray().ravel()
        sims_by_key[key] = rb.aggregate_reference_scores(sims, agg)
    return sims_by_key

MAX_DEV_QUERIES = 1500
b1_valid = train_df.loc[valid_mask & train_df["neutral_mass_estimate"].notna()
                         & train_df["connectivity_key"].notna()].copy()
dev_sample = b1_valid.sample(n=min(MAX_DEV_QUERIES, len(b1_valid)), random_state=SEED)
BEST_PPM = 10

old_bug_rows = []
for agg in ("max", "top3", "mean"):
    ranked_lists, true_labels = [], []
    for pos, qm, true_key in zip(dev_sample.index, dev_sample["neutral_mass_estimate"], dev_sample["connectivity_key"]):
        cand_keys, _, _ = mass_index.rank_by_mass(qm, BEST_PPM)
        sims_by_key = score_candidates_cosine(train_vectors[pos], cand_keys, agg=agg)
        ranked = old_buggy_partition_rank(cand_keys, sims_by_key)
        ranked_lists.append(ranked)
        true_labels.append(true_key)
    m = rb.evaluate_rankings(ranked_lists, true_labels)
    old_bug_rows.append({"experiment": f"OLD_BUGGY_{agg}", **m})

mass_only_ranked = [mass_index.rank_by_mass(qm, BEST_PPM)[0][:25] for qm in dev_sample["neutral_mass_estimate"]]
mass_only_true = dev_sample["connectivity_key"].tolist()
old_bug_rows.insert(0, {"experiment": "B1_mass_only", **rb.evaluate_rankings(mass_only_ranked, mass_only_true)})

old_bug_reproduction = pd.DataFrame(old_bug_rows)
display(old_bug_reproduction)
print("""
Expected pattern reproducing the reported symptom: every OLD_BUGGY_* row scores well below
B1_mass_only, and the OLD_BUGGY_* rows are close to each other regardless of aggregation --
because the hard has-reference/no-reference partition (not the aggregation choice) dominates.
""")


## 7. Corrected hybrid scorer

Switching to `src.retrieval_baseline.hybrid_candidate_score`: an additive blend where a
candidate's mass fit always contributes, and spectral similarity (when a reference spectrum
exists) adds to it -- no hard partition.


In [ ]:
def hybrid_rank(query_pos, query_mass, ppm, agg="max", w_mass=1.0, w_spectral=1.0, ppm_scale=10.0):
    cand_keys, used_ppm, ppm_err = mass_index.rank_by_mass(query_mass, ppm)
    if not cand_keys:
        return [], used_ppm
    has_ref = np.zeros(len(cand_keys), dtype=bool)
    spectral_score = np.zeros(len(cand_keys), dtype=float)
    qvec = train_vectors[query_pos]
    for i, key in enumerate(cand_keys):
        pos_list = conn_to_positions.get(key)
        if pos_list:
            sims = (ref_vectors[np.asarray(pos_list)] @ qvec.T).toarray().ravel()
            spectral_score[i] = rb.aggregate_reference_scores(sims, agg)
            has_ref[i] = True
    scores = rb.hybrid_candidate_score(ppm_err, spectral_score, has_ref,
                                        w_mass=w_mass, w_spectral=w_spectral, ppm_scale=ppm_scale)
    order = np.argsort(-scores, kind="stable")
    return [cand_keys[i] for i in order][:25], used_ppm

hybrid_ranked, hybrid_true = [], []
t0 = time.time()
for pos, qm, true_key in zip(dev_sample.index, dev_sample["neutral_mass_estimate"], dev_sample["connectivity_key"]):
    ranked, _ = hybrid_rank(pos, qm, BEST_PPM, agg="max")
    hybrid_ranked.append(ranked)
    hybrid_true.append(true_key)
hybrid_ms = 1000 * (time.time() - t0) / len(dev_sample)

hybrid_dev_metrics = rb.evaluate_rankings(hybrid_ranked, hybrid_true)
print(f"HYBRID (corrected) on DEV_FOLD sample: {hybrid_dev_metrics}  mean_query_time_ms={hybrid_ms:.2f}")
print(f"B1_mass_only for comparison: {rb.evaluate_rankings(mass_only_ranked, mass_only_true)}")
HYBRID_BEATS_MASS_DEV = hybrid_dev_metrics["mrr_at_25"] >= rb.evaluate_rankings(mass_only_ranked, mass_only_true)["mrr_at_25"]
print(f"\nHybrid MRR@25 >= mass-only MRR@25 on this dev sample: {HYBRID_BEATS_MASS_DEV}")


## 8. Candidate-order-change audit

Guardrail: reranking must actually change candidate order for a meaningful fraction of queries
-- if `fraction_changed == 0`, the "hybrid" scorer is a no-op in disguise (e.g. spectral term
never contributes, or ties are broken identically to mass order).


In [ ]:
change_sample = b1_valid.sample(n=min(300, len(b1_valid)), random_state=SEED)
n_changed = 0
change_rows = []
for pos, qm, true_key in zip(change_sample.index, change_sample["neutral_mass_estimate"], change_sample["connectivity_key"]):
    mass_order = mass_index.rank_by_mass(qm, BEST_PPM)[0][:25]
    hybrid_order, _ = hybrid_rank(pos, qm, BEST_PPM, agg="max")
    changed = mass_order != hybrid_order
    n_changed += int(changed)
    change_rows.append({"changed": changed, "true_in_mass_order": true_key in mass_order,
                         "true_in_hybrid_order": true_key in hybrid_order})

ranking_change_audit = pd.DataFrame(change_rows)
fraction_changed = ranking_change_audit["changed"].mean()
print(f"Fraction of {len(ranking_change_audit)} sampled queries whose top-25 order changed: {fraction_changed:.4f}")
RANKING_CHANGES_OK = bool(fraction_changed > 0)
assert RANKING_CHANGES_OK, "GUARDRAIL FAILED: reranking never changes candidate order -- it is a no-op."
ranking_change_audit.to_csv(OUT_DIR / "ranking_change_audit.csv", index=False)


## 9. Corrected hybrid scorer — full 5-fold evaluation

Re-measures B1 (mass-only) and the corrected hybrid scorer across all 5 folds, plus the
known-vs-novel-reference performance split.


In [ ]:
def set_fold_reference(fold):
    global train_mask_idx, ref_vectors, conn_to_positions
    tm = foldable & (train_df["fold"] != fold)
    train_mask_idx = np.flatnonzero(tm.to_numpy())
    ref_conn = train_df["connectivity_key"].to_numpy()[train_mask_idx]
    conn_to_positions = defaultdict(list)
    for pos, key in enumerate(ref_conn):
        if isinstance(key, str):
            conn_to_positions[key].append(pos)
    ref_vectors = train_vectors[train_mask_idx]

FOLD_MAX_QUERIES = 800
fold_rows = []
known_novel_rows = []
for fold in sorted(train_df["fold"].dropna().unique()):
    fold = int(fold)
    set_fold_reference(fold)
    vmask = train_df["fold"] == fold
    q_df = train_df.loc[vmask & train_df["neutral_mass_estimate"].notna() & train_df["connectivity_key"].notna()]
    if len(q_df) > FOLD_MAX_QUERIES:
        q_df = q_df.sample(n=FOLD_MAX_QUERIES, random_state=SEED)

    mass_ranked = [mass_index.rank_by_mass(qm, BEST_PPM)[0][:25] for qm in q_df["neutral_mass_estimate"]]
    hybrid_ranked_f, in_ref_flags = [], []
    for pos, qm in zip(q_df.index, q_df["neutral_mass_estimate"]):
        ranked, _ = hybrid_rank(pos, qm, BEST_PPM, agg="max")
        hybrid_ranked_f.append(ranked)
    true_f = q_df["connectivity_key"].tolist()
    in_ref_flags = [k in conn_to_positions for k in true_f]

    fold_rows.append({"experiment": "B1_mass_only", "fold": fold, **rb.evaluate_rankings(mass_ranked, true_f)})
    fold_rows.append({"experiment": "HYBRID_corrected", "fold": fold, **rb.evaluate_rankings(hybrid_ranked_f, true_f)})

    for label, mask_fn in [("all", lambda f: [True] * len(f)),
                            ("known_reference", lambda f: f),
                            ("novel_reference", lambda f: [not x for x in f])]:
        sel = mask_fn(in_ref_flags)
        sub_ranked = [r for r, s in zip(hybrid_ranked_f, sel) if s]
        sub_true = [t for t, s in zip(true_f, sel) if s]
        if sub_true:
            known_novel_rows.append({"fold": fold, "group": label, "n": len(sub_true),
                                      **rb.evaluate_rankings(sub_ranked, sub_true)})
    print(f"fold {fold}: {len(q_df)} queries done.")

hybrid_results = pd.DataFrame(fold_rows)
hybrid_summary = hybrid_results.drop(columns=["fold"]).groupby("experiment").agg(["mean", "std"])
display(hybrid_summary)
hybrid_results.to_csv(OUT_DIR / "hybrid_results.csv", index=False)

known_vs_novel_results = pd.DataFrame(known_novel_rows)
display(known_vs_novel_results)
known_vs_novel_results.to_csv(OUT_DIR / "known_vs_novel_results.csv", index=False)

B1_MEAN_MRR = hybrid_summary.loc["B1_mass_only", ("mrr_at_25", "mean")]
HYBRID_MEAN_MRR = hybrid_summary.loc["HYBRID_corrected", ("mrr_at_25", "mean")]
HYBRID_STD_MRR = hybrid_summary.loc["HYBRID_corrected", ("mrr_at_25", "std")]
folds_beating_b1 = (hybrid_results[hybrid_results["experiment"] == "HYBRID_corrected"]
                     .set_index("fold")["mrr_at_25"] >
                     hybrid_results[hybrid_results["experiment"] == "B1_mass_only"].set_index("fold")["mrr_at_25"]).sum()
n_folds = hybrid_results["fold"].nunique()
print(f"\nB1 mean MRR@25: {B1_MEAN_MRR:.4f}")
print(f"Hybrid mean MRR@25: {HYBRID_MEAN_MRR:.4f} +/- {HYBRID_STD_MRR:.4f}")
print(f"Folds where hybrid beats B1: {folds_beating_b1} / {n_folds}")
BEATS_MASS_REFERENCE = bool(HYBRID_MEAN_MRR >= 0.295)  # the project's recorded mass-only reference MRR
NOT_SINGLE_FOLD_ARTIFACT = bool(folds_beating_b1 >= n_folds - 1)
print(f"Hybrid mean MRR@25 >= 0.295 recorded mass-only reference: {BEATS_MASS_REFERENCE}")
print(f"Gain not confined to a single anomalous fold (beats B1 in >= {n_folds-1}/{n_folds} folds): {NOT_SINGLE_FOLD_ARTIFACT}")


## 10. Debug summary

In [ ]:
checks = {
    "cosine_self_similarity_ok": SELF_SIM_OK,
    "cosine_direction_ok": DIRECTION_OK,
    "aggregation_ok": AGGREGATION_OK,
    "multimer_mass_formula_ok": MULTIMER_OK,
    "label_consistency_ok": LABEL_CONSISTENCY_OK,
    "ranking_actually_changes": RANKING_CHANGES_OK,
    "hybrid_beats_mass_on_dev_sample": HYBRID_BEATS_MASS_DEV,
    "hybrid_beats_0.295_reference_meanMRR": BEATS_MASS_REFERENCE,
    "gain_not_single_fold_artifact": NOT_SINGLE_FOLD_ARTIFACT,
}
ALL_PASS = all(checks.values())

summary_lines = ["# Notebook 02b debug summary", ""]
summary_lines.append(f"**Overall status: {'PASS' if ALL_PASS else 'FAIL'}**")
summary_lines.append("")
summary_lines.append("| check | result |")
summary_lines.append("|---|---|")
for k, v in checks.items():
    summary_lines.append(f"| {k} | {'PASS' if v else 'FAIL'} |")
summary_lines.append("")
summary_lines.append(f"- B1 (mass-only) mean MRR@25: {B1_MEAN_MRR:.4f}")
summary_lines.append(f"- HYBRID (corrected) mean MRR@25: {HYBRID_MEAN_MRR:.4f} +/- {HYBRID_STD_MRR:.4f}")
summary_lines.append(f"- Folds where hybrid beats B1: {folds_beating_b1} / {n_folds}")
summary_lines.append(f"- target_in_reference_library rate (DEV_FOLD): {target_in_reference_library.mean():.4%}")
summary_lines.append("")
summary_lines.append("## Root cause")
summary_lines.append(
    "Notebook 02's `cosine_retrieve_hier` partitioned candidates into has-reference-spectrum "
    "(ranked first, however weak) vs. no-reference-spectrum (mass-order fallback, ranked after "
    "ALL reference-bearing candidates). Under connectivity-grouped CV the true candidate almost "
    "always lacks a reference spectrum, so it was systematically buried below any wrong "
    "candidate with even a weak cosine score -- explaining both the sub-mass-only scores and "
    "the near-identical B2-B6 results (conditioning only reshuffled the small reference-bearing "
    "subset, never the dominant partition)."
)
summary_lines.append("")
summary_lines.append("## Fix")
summary_lines.append(
    "`src/retrieval_baseline.hybrid_candidate_score`: an additive blend "
    "`w_mass * mass_score_from_ppm(abs_ppm_error) + w_spectral * spectral_score` where "
    "`spectral_score` is 0 (not a partition exclusion) when no reference spectrum exists -- mass "
    "fit always contributes to the rank."
)

debug_summary_path = OUT_DIR / "debug_summary.md"
debug_summary_path.write_text("\n".join(summary_lines), encoding="utf-8")
print("\n".join(summary_lines))
print(f"\nSaved {debug_summary_path}")

print(f"\n{'='*60}\nPRECONDITION STATUS FOR NOTEBOOK 03: {'PASS' if ALL_PASS else 'FAIL'}\n{'='*60}")
if not ALL_PASS:
    failed = [k for k, v in checks.items() if not v]
    print("Failed checks (Notebook 03 must not proceed until these pass):")
    for k in failed:
        print(f"  - {k}")
